# FMCG Demand Forecasting & Inventory Optimization

Independent project using the public Walmart M5 retail dataset as an FMCG proxy.

**Business objective:** forecast SKU-store demand for 28 days and translate the forecast into replenishment decisions.


## 1. Project design

Workflow: data cleaning → EDA → representative SKU-store selection → baseline/statistical/ML forecasting → chronological validation → 28-day out-of-sample evaluation → safety stock → reorder point.

The raw M5 CSV files are intentionally kept outside GitHub. Place them under `data/raw/`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from scipy.stats import norm

DATA = Path('../data/raw')
sales = DATA / 'sales_train_validation.csv'
evaluation = DATA / 'sales_train_evaluation.csv'
calendar_file = DATA / 'calendar.csv'
prices_file = DATA / 'sell_prices.csv'


## 2. Load and inspect the data


In [ ]:
calendar = pd.read_csv(calendar_file, parse_dates=['date'])
meta = pd.read_csv(sales, usecols=['id','item_id','dept_id','cat_id','store_id','state_id'])
print('SKU-store series:', len(meta))
print('Items:', meta.item_id.nunique())
print('Stores:', meta.store_id.nunique())
print('Categories:', meta.cat_id.nunique())
print('Historical days:', sum(c.startswith('d_') for c in pd.read_csv(sales, nrows=0).columns))


## 3. Data quality checks

The uploaded project data was checked for missing sales values, negative values, non-integer observations and duplicate IDs. Calendar event nulls were made explicit and price records were cleaned for missing/non-positive/duplicate observations.


In [ ]:
sales_check = pd.read_csv(sales)
day_cols = [c for c in sales_check.columns if c.startswith('d_')]
values = sales_check[day_cols].to_numpy()
print('Missing sales:', np.isnan(values).sum())
print('Negative sales:', (values < 0).sum())
print('Non-integer sales:', (values != np.floor(values)).sum())
print('Duplicate IDs:', sales_check.id.duplicated().sum())


## 4. Exploratory Data Analysis

The historical unit mix is approximately FOODS 68.6%, HOUSEHOLD 22.0%, HOBBIES 9.3%. About 68.2% of SKU-store-day observations are zero, indicating substantial intermittent demand.


In [ ]:
cat_sales = sales_check.groupby('cat_id')[day_cols].sum().sum(axis=1).sort_values(ascending=False)
print((cat_sales / cat_sales.sum() * 100).round(1))
zero_share = (values == 0).mean()
print(f'Overall zero-sales share: {zero_share:.2%}')


## 5. Forecasting setup

Use a chronological split: first 1,885 days for training and the final 28 historical days for validation. Model selection never uses the future evaluation file. After selection, refit on all 1,913 historical days and evaluate on the next 28 actual days.

Candidate models: Naive, 7-day Seasonal Naive, 7-day Moving Average, Damped Holt-Winters, and Gradient Boosting using lag/rolling/calendar features.


In [ ]:
def mae(y, yhat):
    return mean_absolute_error(y, yhat)

def rmse(y, yhat):
    return mean_squared_error(y, yhat, squared=False)

def seasonal_naive(y, horizon=28, season=7):
    base = np.asarray(y[-season:])
    return np.resize(base, horizon)

def moving_average(y, horizon=28, window=7):
    return np.repeat(np.mean(y[-window:]), horizon)


## 6. Inventory decision layer

Because the public dataset has no true supplier lead time or on-hand inventory, the inventory layer uses explicit assumptions: 7-day lead time and 95% service level.

Lead-time demand = average forecast demand × lead time.

Safety stock = z × demand variability × √lead time, where z ≈ 1.645 for 95% service.

Reorder point = lead-time demand + safety stock.


In [ ]:
LEAD_TIME_DAYS = 7
SERVICE_LEVEL = 0.95
Z = norm.ppf(SERVICE_LEVEL)

def inventory_policy(forecast, demand_history, lead_time=LEAD_TIME_DAYS):
    avg_demand = float(np.mean(forecast))
    residual_scale = float(np.std(np.diff(np.asarray(demand_history))))
    lead_demand = avg_demand * lead_time
    safety_stock = Z * residual_scale * np.sqrt(lead_time)
    reorder_point = lead_demand + safety_stock
    return {'avg_daily_forecast': avg_demand, 'lead_time_demand': lead_demand, 'safety_stock': safety_stock, 'reorder_point': reorder_point}


## 7. Project results

On the selected seven-series panel, mean validation MAE was 9.70 for the 7-day moving average, 10.43 for Gradient Boosting, 11.59 for seasonal naive, 16.07 for Holt-Winters and 24.07 for naive. The selected per-series models achieved mean validation MAE of 8.41.

On the 28-day out-of-sample evaluation: MAE 7.99, RMSE 9.82 and WAPE 39.25%.

These results apply only to the selected representative panel, not to the full M5 dataset.


## 8. Business interpretation

The key supply-chain takeaway is that forecasting should support an inventory decision. Different SKU-store series can require different forecasting approaches, intermittent demand needs careful evaluation, and service-level choices directly affect safety stock and working capital.
